# 01 — Run Experiments (pi0.5, LIBERO / LIBERO-PRO)

**This notebook IS the experiment.** The `pnp` package provides primitives
(`run_episode`, `iter_task_envs`, `store`); the loop + the `METHODS` dict below are the
visible spec of what runs. Edit the flags, re-run. Results go to Supabase.

## 1. Fresh GPU runtime: secrets + deterministic install

Start from a **fresh Colab GPU runtime**. This installs the pinned pi0.5 stack while preserving Colab's native Torch/TorchVision/CUDA packages.

In [ ]:
import os
import subprocess
import sys
from google.colab import userdata

for key in ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "HF_TOKEN"):
    os.environ[key] = userdata.get(key)

GH_PAT = userdata.get("GH_PAT")
REPO_DIR = "/content/cs159-sp26"
GIT_REF = "main"
REPO_URL = f"https://{GH_PAT}@github.com/ArjunS07/cs159-sp26.git"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", "--branch", GIT_REF, REPO_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", GIT_REF], check=True)
    subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", GIT_REF],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        f"{REPO_DIR}/pnp-vla[sim]",
    ],
    check=True,
)

import pnp
print("Loaded pnp from:", pnp.__file__)

## 2. Environment + model + store

In [ ]:
from pnp.env_setup import setup_environment
setup_environment()

In [ ]:
import contextlib
import io

from pnp import libero_env, models, RolloutConfig, Method
from pnp.store import SupabaseStore
from pnp.rollout import run_episode, iter_task_envs
from tqdm.auto import tqdm

# LIBERO/robosuite emits an INFO line for every task it enumerates. Capture only that noisy
# setup output; model download progress, exceptions, and rollout progress remain visible.
libero_setup_output = io.StringIO()
with contextlib.redirect_stdout(libero_setup_output), contextlib.redirect_stderr(libero_setup_output):
    benchmark_dict = libero_env.init_libero_benchmark()

policy, preprocess, postprocess = models.load_pi05()
device = models.default_device()
store = SupabaseStore()

LIBERO_SUITES = ('libero_spatial', 'libero_object', 'libero_goal', 'libero_10')
with contextlib.redirect_stdout(libero_setup_output), contextlib.redirect_stderr(libero_setup_output):
    libero_tasks = [
        (suite, task_idx)
        for suite in LIBERO_SUITES
        for task_idx in range(benchmark_dict[suite]().n_tasks)
    ]
    episodes = libero_env.build_final_episodes(benchmark_dict, tasks=libero_tasks)

assert len(episodes) == 400, f'expected 400 LIBERO identities, got {len(episodes)}'
print(f'Prepared {len(episodes)} LIBERO episode identities across {len(libero_tasks)} tasks.')

## 3. Full LIBERO schedule ablation (400 identities)

Eight refinement schedules run without screening at fixed `K=3`. One shared observed rollout
measures their union of Euler steps; the three `extra_steps` arms match the unique compute costs.

In [ ]:
SCHEDULES = (
    (2, 3), (3, 4), (4, 5), (5, 6), (7, 8),
    (1, 3, 5, 7, 9), (3, 6, 9), (2, 5, 8),
)
K = 3
BASE_INFERENCE_STEPS = 10
OBSERVED_STEPS = tuple(sorted({step for schedule in SCHEDULES for step in schedule}))

def build_schedule_methods(schedules=SCHEDULES, k=K):
    extra_steps = sorted({BASE_INFERENCE_STEPS + k * len(schedule) for schedule in schedules})
    methods = [
        (Method.UNCERTAINTY, RolloutConfig(
            pnp_steps=OBSERVED_STEPS, pnp_k=k, save_pcp_features=True)),
        *[
        (Method.EXTRA_STEPS, RolloutConfig(num_inference_steps=steps))
        for steps in extra_steps
        ],
    ]
    for schedule in schedules:
        probe = dict(pnp_steps=schedule, pnp_k=k)
        methods.extend([
            (Method.REFINEMENT, RolloutConfig(**probe, refine=True)),
            (Method.REFINEMENT, RolloutConfig(
                **probe, refine=True, refine_average=True)),
        ])
    assert len(methods) == 20
    logical_hashes = {
        store.config_hash(store._logical_key(name, cfg)) for name, cfg in methods
    }
    assert len(logical_hashes) == len(methods), 'rollout configs must hash uniquely'
    return methods

METHODS = build_schedule_methods()

def run_collection(experiment, benchmark, collection_episodes, methods=METHODS):
    expected = len(collection_episodes) * len(methods)
    print(f'{benchmark}: {len(collection_episodes)} identities x {len(methods)} configs = {expected} rollouts')
    store.start_run(
        driver='full_schedules', benchmark=benchmark, experiment=experiment,
        config={'schedules': SCHEDULES, 'pnp_k': K, 'n_configs': len(methods)},
    )
    done = store.existing_keys(experiment)
    pending = max(expected - len(done), 0)
    n = 0
    with tqdm(total=pending, desc=benchmark, unit='rollout', dynamic_ncols=True) as progress:
        for env, task_eps in iter_task_envs(collection_episodes):
            for ep, name, cfg, rid in store.iter_todo(experiment, task_eps, methods, done=done):
                res = run_episode(env, ep, policy, preprocess, device, cfg)
                store.log_result(rid, ep, name, cfg, res)
                n += 1
                progress.update()
                progress.set_postfix(success=int(res['success']), method=name, refresh=False)
    store.finish_run(n_rollouts=n)
    print(f'logged {n} new rollouts to experiment={experiment}  (~{store.bytes_written/1e6:.1f} MB blobs)')

EXPERIMENT = 'libero-full-schedules-k3-v1'
run_collection(EXPERIMENT, 'libero', episodes)

## 4. Deduplicated LIBERO-PRO union

Uses the same 20 configurations and fixed `K=3`. Run the LIBERO-PRO setup notebook first,
then set the gate below to `True`. The manifest retains canonical/expanded cohort metadata.

In [ ]:
RUN_LIBERO_PRO = False

if RUN_LIBERO_PRO:
    from pnp import libero_pro

    libero_pro.apply_env_patches()
    libero_pro.patch_torch_load()
    pro_benchmark_dict = libero_pro.reload_benchmark()
    pro_episodes = libero_pro.build_libero_pro_episodes(pro_benchmark_dict)
    pro_keys = {
        (ep['suite'], ep['task_idx'], ep['ep_idx'], ep['init_state_hash'])
        for ep in pro_episodes
    }
    assert len(pro_keys) == len(pro_episodes), 'PRO manifest contains duplicate identities'
    assert all(
        ep['canonical_member'] or ep['expanded_member'] for ep in pro_episodes
    )
    PRO_EXPERIMENT = 'pro-union-full-schedules-k3-v1'
    run_collection(PRO_EXPERIMENT, 'libero_pro', pro_episodes)
else:
    print('LIBERO-PRO skipped; run its setup notebook, then set RUN_LIBERO_PRO=True.')